In [ ]:
!pip install -U transformers datasets evaluate accelerate
!nvidia-smi

example_text = "Maximum Inner Product Search (MIPS) is a ubiquitous task in " \
+ "machine learning applications such as recommendation systems. Given a query " \
+ "vector and n atom vectors in d-dimensional space, the goal of MIPS is to " \
+ "find the atom that has the highest inner product with the query vector"

from transformers import AutoTokenizer, AutoModelWithLMHead

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")


# vars(model.transformer).keys()
# vars(model.lm_head).keys()

import torch

model.lm_head.weight.size()
norm = torch.norm(model.lm_head.weight, dim=1)
print(f"max: {norm.max()}/ min: {norm.min()}")

import os

def generate_queries():
  input_ids = tokenizer.encode(
            example_text,
            return_tensors="pt",
        )  # Encode input sequence
  output = model.transformer(input_ids=input_ids)

  data_dir = "mips_gpt2"
  os.makedirs(data_dir, exist_ok=True)
  torch.save(model.lm_head.weight, os.path.join(data_dir, "GPT2_LM_HEAD_WEIGHT.pt"))
  torch.save(output.last_hidden_state[-10:], os.path.join(data_dir,"GPT2_QUERIES.pt"))

generate_queries()